In [1]:
import pymupdf

doc = pymupdf.open("../data/raw/selling_guide.pdf")
print(doc.page_count)

toc = doc.get_toc()
print(len(toc))
print(toc[:20])

1196
559
[[1, 'Table of Contents', 3], [1, 'Preface', 14], [1, 'Part A, Doing Business with Fannie Mae', 19], [2, 'Subpart A1, Approval Qualification', 20], [3, 'Chapter A1-1, Application and Approval of Seller/Servicer', 20], [4, 'A1-1-01, Application and Approval of Seller/Servicer (09/04/2018)', 20], [2, 'Subpart A2, Lender Contract', 20], [3, 'Chapter A2-1, Contractual Obligations for Sellers/Servicers', 21], [4, 'A2-1-01, Contractual Obligations for Sellers/Servicers (06/04/2025)', 21], [4, 'A2-1-02, Nature of Mortgage Transaction (02/27/2018)', 25], [4, 'A2-1-03, Indemnification for Losses (10/06/2021)', 26], [3, 'Chapter A2-2, Contractual Representations and Warranties', 31], [4, 'A2-2-01, Representations and Warranties Overview (02/05/2020)', 32], [4, 'A2-2-02, Delivery Information and Delivery-Option Specific Representations and Warranties (08/30/2016)', 34], [4, 'A2-2-03, Document Warranties (10/06/2021)', 35], [4, 'A2-2-04, Limited Waiver and Enforcement Relief of Representa

In [2]:
start = end = None

for level, title, page in toc:
    if level == 2 and title.startswith("Subpart B3"):
        start = page
    elif level == 2 and title.startswith("Subpart B4"):
        end = page
        break

print(start, end)

285 544


In [3]:
from collections import Counter

# Keep only entries that belong to Part B3
b3_entries = [
    (level, title, page)
    for level, title, page in toc
    if title.startswith("Subpart B3")
    or title.startswith("Chapter B3-")
    or title.startswith("B3-")
]


level_counts = Counter(level for level, title, page in b3_entries)
print("B3 entries by level:", dict(level_counts))
print("Highest level in toc:", max(level for level, title, page in toc))

B3 entries by level: {2: 1, 3: 6, 4: 20, 5: 103}
Highest level in toc: 5


In [4]:
import re

# Matches rule titles like:
#   "B3-6-02, Debt-to-Income Ratios (04/02/2025)"
#   "B3-3.3-01, Base Income (03/04/2026)"
RULE_PATTERN = re.compile(
    r"^(B3-\d+(?:\.\d+)?-\d+),\s*(.+?)\s*\((\d{2}/\d{2}/\d{4})\)$"
)

rules = []
for level, title, page in toc:
    match = RULE_PATTERN.match(title.strip())
    if match:
        rule_id, rule_title, updated = match.groups()
        rules.append({
            "rule_id": rule_id,
            "title": rule_title,
            "updated": updated,
            "chapter": re.match(r"B3-\d+", rule_id).group(),
            "start_page": page,
        })

# Each rule ends where the next one starts; the last one ends at B4
for i, rule in enumerate(rules):
    if i + 1 < len(rules):
        rule["end_page"] = rules[i + 1]["start_page"]
    else:
        rule["end_page"] = end

print(len(rules))
print(rules[0])
print(next(r for r in rules if r["rule_id"] == "B3-6-02"))

123
{'rule_id': 'B3-1-01', 'title': 'Comprehensive Risk Assessment', 'updated': '11/05/2025', 'chapter': 'B3-1', 'start_page': 286, 'end_page': 288}
{'rule_id': 'B3-6-02', 'title': 'Debt-to-Income Ratios', 'updated': '04/02/2025', 'chapter': 'B3-6', 'start_page': 524, 'end_page': 528}


In [5]:
rule = next(r for r in rules if r["rule_id"] == "B3-6-02")

for page_num in range(rule["start_page"], rule["end_page"] + 1):
    page = doc[page_num - 1]  # PyMuPDF counts pages from 0, the TOC counts from 1
    print(f"\n========== PAGE {page_num} ==========")
    print(page.get_text())


========== PAGE 524 ==========
Published September 2, 2026
506
union dues; and
voluntary deductions.
Recent Related Announcements
The table below provides references to recently issued Announcements that are related to this topic.
Announcement
Issue Date
Announcement SEL-2022-04
May 04, 2022
Announcement SEL-2020-01
February 05, 2020
 
B3-6-02, Debt-to-Income Ratios (04/02/2025)
Introduction
This topic contains information on the use of the debt-to-income (DTI) ratio, including:
DTI Ratios
Maximum DTI Ratios
Exceptions to the Maximum DTI Ratio
Calculating Total Monthly Obligation
DTI Ratio Tolerance and Re-Underwriting Criteria
Applying the Re-underwriting Criteria
DTI Ratios
The DTI ratio consists of two components:
total monthly obligations, which includes the qualifying payment for the subject mortgage loan and other
long-term and signiﬁcant short-term monthly debts (see Calculating Total Monthly Obligation below); and
total monthly income of all borrowers, to the extent the income

In [6]:
import re
import unicodedata

HEADER_PATTERN = re.compile(r"^Published \w+ \d{1,2}, \d{4}$")

def clean_page_text(text):
    """Clean the raw text of one PDF page.

    Returns (cleaned_text, printed_page_number).
    """
    # 1. Fix ligatures like "ﬁ" -> "fi"
    text = unicodedata.normalize("NFKC", text)

    lines = text.splitlines()
    printed_page = None

    # 2. Remove the "Published ..." header 
    if lines and HEADER_PATTERN.match(lines[0].strip()):
        lines = lines[1:]

        # 3. Remove the printed page number right after the header
        if lines and lines[0].strip().isdigit():
            printed_page = int(lines[0].strip())
            lines = lines[1:]

    return "\n".join(lines).strip(), printed_page


# Test on page 524
cleaned, printed = clean_page_text(doc[524 - 1].get_text())
print("Printed page:", printed)
print(cleaned[:300])

Printed page: 506
union dues; and
voluntary deductions.
Recent Related Announcements
The table below provides references to recently issued Announcements that are related to this topic.
Announcement
Issue Date
Announcement SEL-2022-04
May 04, 2022
Announcement SEL-2020-01
February 05, 2020
 
B3-6-02, Debt-to-Income R


In [7]:
pages = []
for pdf_page in range(start, end + 1):
    text, printed = clean_page_text(doc[pdf_page - 1].get_text())
    pages.append({
        "pdf_page": pdf_page,
        "printed_page": printed,
        "text": text,
    })

print("Pages cleaned:", len(pages))

# Check: did every page have a header and page number?
missing = [p["pdf_page"] for p in pages if p["printed_page"] is None]
print("Pages with no printed page number:", missing)

offsets = {p["pdf_page"] - p["printed_page"] for p in pages if p["printed_page"]}
print("Page offsets found:", offsets)

leftovers = [p["pdf_page"] for p in pages if "ﬁ" in p["text"] or "Published" in p["text"]]
print("Pages with leftovers:", leftovers)

Pages cleaned: 260
Pages with no printed page number: []
Page offsets found: {18}
Pages with leftovers: []


In [11]:
import bisect
import re
import unicodedata

# Join all pages into one text, remembering where each page starts
full_text = ""
page_starts = []
for p in pages:
    page_starts.append(len(full_text))
    full_text += p["text"] + "\n"

def page_at(position):
    """Return the page that contains a given character position."""
    i = bisect.bisect_right(page_starts, position) - 1
    return pages[i]

def heading_pattern(rule):
    """Regex for a rule's full heading; allows line breaks between words."""
    title = unicodedata.normalize("NFKC", rule["title"])
    title_regex = r"\s+".join(
    re.escape(word).replace(r"\-", r"-\s*") for word in title.split()
)
    return re.compile(
        rf"^{re.escape(rule['rule_id'])},\s+{title_regex}\s*\({re.escape(rule['updated'])}\)",
        re.MULTILINE,
    )

# Find each heading, searching forward from the previous one
positions = []
search_from = 0
for rule in rules:
    match = heading_pattern(rule).search(full_text, search_from)
    if match is None:
        print("NOT FOUND:", rule["rule_id"], rule["title"])
        positions.append(None)
        continue
    positions.append(match.start())
    search_from = match.end()

print("Headings found:", sum(p is not None for p in positions), "of", len(rules))

Headings found: 123 of 123


In [13]:
for i, rule in enumerate(rules):
    start_pos = positions[i]
    end_pos = positions[i + 1] if i + 1 < len(rules) else len(full_text)
    text = full_text[start_pos:end_pos]

    # The last rule runs to the end of page 544, which may include the start of B4
    if i + 1 == len(rules):
        b4 = text.find("Subpart B4")
        if b4 != -1:
            text = text[:b4]

    # Drop the "Recent Related Announcements" table at the end of the rule
    announcements = text.find("Recent Related Announcements")
    if announcements != -1:
        text = text[:announcements]

    rule["text"] = text.strip()
    rule["printed_start"] = page_at(start_pos)["printed_page"]
    rule["printed_end"] = page_at(start_pos + len(text) - 1)["printed_page"]

# Checks
word_counts = sorted((len(r["text"].split()), r["rule_id"]) for r in rules)
print("Shortest rules:", word_counts[:3])
print("Longest rules:", word_counts[-3:])

b3_6_02 = next(r for r in rules if r["rule_id"] == "B3-6-02")
print("\nB3-6-02 printed pages:", b3_6_02["printed_start"], "to", b3_6_02["printed_end"])
print(b3_6_02["text"][:300])
print("...")
print(b3_6_02["text"][-300:])

Shortest rules: [(50, 'B3-4.3-07'), (59, 'B3-2-08'), (59, 'B3-4.3-17')]
Longest rules: [(2584, 'B3-6-05'), (2871, 'B3-2-02'), (3703, 'B3-5.3-09')]

B3-6-02 printed pages: 506 to 509
B3-6-02, Debt-to-Income Ratios (04/02/2025)
Introduction
This topic contains information on the use of the debt-to-income (DTI) ratio, including:
DTI Ratios
Maximum DTI Ratios
Exceptions to the Maximum DTI Ratio
Calculating Total Monthly Obligation
DTI Ratio Tolerance and Re-Underwriting Criteria
Ap
...
nclude all income and debts verified,
disclosed, or identified during the mortgage process.
6
Upon delivery to Fannie Mae, the lender must deliver the qualifying monthly income and expense
amounts that are on the final loan application. See C1-2-02, Loan Data and Documentation
Delivery Requirements.


In [14]:
import json
from pathlib import Path

out_dir = Path("../data/processed")
out_dir.mkdir(parents=True, exist_ok=True)

with open(out_dir / "b3_rules.json", "w") as f:
    json.dump(rules, f, indent=2)

print("Saved", len(rules), "rules")

Saved 123 rules


In [15]:
type(rule)

dict

In [17]:
rule

{'rule_id': 'B3-6-08',
 'title': 'DU: Requirements for Liability Assessment',
 'updated': '01/27/2015',
 'chapter': 'B3-6',
 'start_page': 543,
 'end_page': 544,
 'text': 'B3-6-08, DU: Requirements for Liability Assessment (01/27/2015)\nIntroduction\nThis topic contains information on DU requirements for liability assessment, including:\nReconciling the Loan Application with the Credit Report\nAuto-Populating DU Liabilities from the Credit Report\nReconciling the Loan Application with the Credit Report\nDU uses liabilities from the loan application, not debts from the credit report, to calculate the debt-to-income\nratio.\nTo help ensure that all appropriate liabilities are included in the debt-to-income ratio, DU performs a series of\nreasonableness tests comparing loan application balances and payments with the credit report balances and\npayments. If the values on the loan application are less than the values on the credit report by more than\nselected tolerances, the lender must ju

In [18]:
type(rules)

list

In [19]:
len(rules)

123

In [22]:
rules[-1]

{'rule_id': 'B3-6-08',
 'title': 'DU: Requirements for Liability Assessment',
 'updated': '01/27/2015',
 'chapter': 'B3-6',
 'start_page': 543,
 'end_page': 544,
 'text': 'B3-6-08, DU: Requirements for Liability Assessment (01/27/2015)\nIntroduction\nThis topic contains information on DU requirements for liability assessment, including:\nReconciling the Loan Application with the Credit Report\nAuto-Populating DU Liabilities from the Credit Report\nReconciling the Loan Application with the Credit Report\nDU uses liabilities from the loan application, not debts from the credit report, to calculate the debt-to-income\nratio.\nTo help ensure that all appropriate liabilities are included in the debt-to-income ratio, DU performs a series of\nreasonableness tests comparing loan application balances and payments with the credit report balances and\npayments. If the values on the loan application are less than the values on the credit report by more than\nselected tolerances, the lender must ju